In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import fastf1
from fastf1 import plotting
from src.utils import get_driver_telemetry_for_laps

# Enable FastF1's plotting style
plotting.setup_mpl(color_scheme='fastf1')

In [ ]:
fastf1.Cache.enable_cache('./f1_cache')
fastf1.Cache.get_cache_info()

In [ ]:
race = fastf1.get_session(2025,'Silverstone','Race')

In [ ]:
race.load()

In [ ]:
pia_car_tel = get_driver_telemetry_for_laps(race,'PIA',21,tel_type='car').add_distance()
pia_car_tel['absolute_time'] = pia_car_tel.SessionTime.apply(lambda x: x.total_seconds()) - race.session_start_time.total_seconds()
pia_subset_tel = pia_car_tel[(pia_car_tel.absolute_time < 2725) & (pia_car_tel.absolute_time > 2715)].copy()

In [ ]:
# pia_subset_tel.loc[456,'Brake'] = np.True_
# pia_subset_tel.loc[473,'Brake'] = np.True_
# pia_subset_tel.loc[474,'Brake'] = np.True_

In [ ]:
ver_car_tel = get_driver_telemetry_for_laps(race,'VER',21,tel_type='car').add_distance()
ver_car_tel['absolute_time'] = ver_car_tel.SessionTime.apply(lambda x: x.total_seconds()) - race.session_start_time.total_seconds()
ver_subset_tel = ver_car_tel[(ver_car_tel.absolute_time < 2725) & (ver_car_tel.absolute_time > 2715)].copy()

In [ ]:
# ver_subset_tel.loc[457,'Brake'] = np.True_
# ver_subset_tel.loc[477,'Brake'] = np.True_

In [ ]:
# Get team color
mclaren_color = plotting.get_team_color('McLaren',race)
redbull_color = plotting.get_team_color('RedBull',race)

fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot Speed on primary y-axis
ax1.plot(pia_subset_tel['absolute_time'], pia_subset_tel['Speed'], color=mclaren_color, lw=3, label='PIA Speed')
ax1.plot(ver_subset_tel['absolute_time'], ver_subset_tel['Speed'], color=redbull_color, lw=3, label='VER Speed')

# Create secondary y-axis
ax2 = ax1.twinx()

# Plot Brake on secondary y-axis
ax2.plot(pia_subset_tel['absolute_time'],pia_subset_tel['Brake']*1,color=mclaren_color,alpha=1)
ax2.plot(ver_subset_tel['absolute_time'],ver_subset_tel['Brake']*1,color='w',alpha=1)

# Add shaded area under each line
ax2.fill_between(pia_subset_tel['absolute_time'], pia_subset_tel['Brake'], color=mclaren_color, alpha=0.4,label='PIA Brake')
ax2.fill_between(ver_subset_tel['absolute_time'], ver_subset_tel['Brake'], color='w', alpha=0.2, label='VER Brake')

# Add ticks, labels and limits
ax1.set_ylabel('Speed (km/h)',fontsize=16)
ax1.set_xlabel('Relative Session Time (s)',fontsize=16)
ax1.set_xticks(range(2715,2725))
ax1.set_xticklabels(labels=['45m 15s']+['+'+str(i) for i in range(1,10)],fontsize=14)
ax1.set_yticks(range(0,230,25))
ax1.set_yticklabels(labels=['']+[str(i) for i in range(25,230,25)],fontsize=14)
ax1.set_xlim([2715,2724])
ax1.set_ylim([0,225])

ax2.set_axis_off()
ax2.set_ylim([0,5])

# Add Legends
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
legend1 = ax1.legend(lines_1, labels_1, title='Speed (kmph)',loc=(0.63,0.75),fancybox=True,facecolor="#000000",edgecolor='w',fontsize=14)
legend2 = ax2.legend(lines_2, labels_2, title='Brakes (ON/OFF)',loc=(0.8,0.75),fancybox=True,facecolor="#000000",edgecolor='w',fontsize=14)

legend1.get_title().set_fontsize(16)
legend2.get_title().set_fontsize(16)

# Title
ax1.set_title("Silverstone '25, Lap 21: Safety Car Ending",fontsize=24)

# Show Plot
ax1.grid(color="#333333")
plt.tight_layout()
plt.show()

In [ ]:
df = pia_subset_tel[['Speed','absolute_time']].copy()

In [ ]:
df['long_g_force'] = df.diff().apply(lambda x: (x['Speed']*5/18)/(x['absolute_time']*9.8),axis=1)

In [ ]:
df.long_g_force.min()

In [ ]:
pia_lap17 = get_driver_telemetry_for_laps(race,'PIA',17,tel_type='car').add_distance()
pia_lap17 = pia_lap17[(4000 <= pia_lap17.Distance) & (pia_lap17.Distance <= 5000)].copy()
pia_lap21 = get_driver_telemetry_for_laps(race,'PIA',21,tel_type='car').add_distance()
pia_lap21 = pia_lap21[(4000 <= pia_lap21.Distance) & (pia_lap21.Distance <= 5000)].copy()

In [ ]:
circuit_info = race.get_circuit_info()

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

# Plot Speed on y-axis
ax.plot(pia_lap17['Distance'], pia_lap17['Speed'], lw=3, label='PIA Lap 17')
ax.plot(pia_lap21['Distance'], pia_lap21['Speed'], lw=3, label='PIA Lap 21')

# Add ticks, labels and limits
ax.set_ylabel('Speed (km/h)',fontsize=16)
ax.set_xlabel('Lap Distance Traversed (km)',fontsize=16)
ax.set_xticks(range(4000,5200,200))
ax.set_xticklabels(labels=[i/1000 for i in range(4000,5200,200)],fontsize=14)
ax.set_yticks(range(0,230,25))
ax.set_yticklabels(labels=['']+[str(i) for i in range(25,230,25)],fontsize=14)
ax.set_xlim([4000,5000])
ax.set_ylim([0,225])

# Add legends
lines, labels = ax.get_legend_handles_labels()
legend = ax.legend(lines,labels,title='Speed (kmph)',loc=(0.8,0.06),fancybox=True,facecolor="#000000",edgecolor='w',fontsize=14)
legend.get_title().set_fontsize(16)

# Title
ax.set_title("Silverstone '25, Lap 17 & Lap 21: Safety Car Ending",fontsize=24)

# Show Plot
ax.grid(color="#333333")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Set up figure and axis
fig, ax = plt.subplots(figsize=(12,6))

# Create empty line objects for animation
line1, = ax.plot([], [], lw=3, label='PIA Lap 17')
line2, = ax.plot([], [], lw=3, label='PIA Lap 21')

# Set labels, ticks, limits (from your original code)
ax.set_ylabel('Speed (km/h)', fontsize=16)
ax.set_xlabel('Lap Distance Traversed (km)', fontsize=16)
ax.set_xticks(range(4000, 5200, 200))
ax.set_xticklabels(labels=[i/1000 for i in range(4000, 5200, 200)], fontsize=14)
ax.set_yticks(range(0, 230, 25))
ax.set_yticklabels(labels=[''] + [str(i) for i in range(25, 230, 25)], fontsize=14)
ax.set_xlim([4000, 5000])
ax.set_ylim([0, 225])
ax.set_title("Silverstone '25, Lap 17 & Lap 21: Safety Car Ending", fontsize=24)
ax.grid(color="#333333")

# Add legend
lines, labels = ax.get_legend_handles_labels()
legend = ax.legend(lines, labels, title='Speed (kmph)', loc=(0.8, 0.06),
                   fancybox=True, facecolor="#000000", edgecolor='w', fontsize=14)
legend.get_title().set_fontsize(16)

plt.tight_layout()

# Animation update function
def update(frame):
    # Update both lines up to current frame
    line1.set_data(pia_lap17['Distance'][:frame], pia_lap17['Speed'][:frame])
    line2.set_data(pia_lap21['Distance'][:frame], pia_lap21['Speed'][:frame])
    return line1, line2

# Create animation
ani = FuncAnimation(
    fig,
    update,
    frames=len(pia_lap17),
    interval=20,  # milliseconds between frames (adjust speed here)
    blit=True
)

# Display animation in notebook
HTML(ani.to_jshtml())
